# 05 - 表达式派生与约束求解

> **何时使用**: 当列之间存在计算关系（如 `short_code = project_no[-6:]`），或有 UNIQUE 约束时。
>
> **核心概念**: `derive_from` 声明列依赖，`expression` 指定计算公式，ColumnDAG 自动拓扑排序。

## 适用场景

- 列 B 依赖列 A 的值（如缩写、截取、拼接）→ `derive_from` + `expression`
- 列有 UNIQUE 约束 → ConstraintSolver 自动回溯求解
- 大数据量（>100K 行）UNIQUE → 自动切换概率模式
- 需要链式依赖（A → B → C）→ ColumnDAG 拓扑排序

## 你将学到

- `derive_from` + `expression` 列派生
- ColumnDAG 拓扑排序
- ExpressionEngine 21 个安全函数
- UNIQUE 约束回溯求解
- 概率模式（>100K 行）

详见 architecture.zh-CN.md §6

**📚 教程导航**

| 序号 | 主题 | 架构层 | 前置要求 |
|------|------|--------|----------|
| 01 | 快速上手与核心流程 | Orchestrator | 无 |
| 02 | 9 级策略链详解 | Core: ColumnMapper | 01 |
| 03 | 生成器与 Provider 体系 | Generators | 01 |
| 04 | 数据库层与多表关联 | Database + Core | 01 |
| **→ 05** | **表达式派生与约束求解** | **Core: DAG / Expression** | **01** |
| 06 | 配置驱动与 Transform | Config / Core | 01 |
| 07 | AI 智能配置 | Plugins: AI | 01 |
| 08 | MCP 服务器集成 | Plugins: MCP | 07 |
| 09 | 插件系统与 Hook 生命周期 | Plugins | 01 |
| 10 | CLI 参考手册 | CLI | 06 |
| 11 | 工具类参考 | Utils | 01 |
| 12 | 测试集成模式 | Testing | 01 |

---

In [1]:
PRJ_PATTERN = "PRJ-\\d{6}"
# Prerequisite: pip install -e ".[dev,all]"
import sqlseed
from sqlseed import connect, fill, fill_from_config, preview

# Demo database setup
import sys; sys.path.insert(0, "..")  # for build_demo_db only
from build_demo_db import build
db_path = build()  # Force rebuild to ensure idempotent run

# Populate base dependencies
with connect(str(db_path)) as orch:
    orch.fill_table("organizations", count=5, seed=42)
    orch.fill_table("members", count=20, seed=42)
    orch.fill_table("projects", count=10, seed=42)
    orch.fill_table("tags", count=8, seed=42)

print(f"sqlseed {sqlseed.__version__} | Database: {db_path}")

Generating organizations:   0%|          | 0/5 [00:00<?, ?it/s]

Generating members:   0%|          | 0/20 [00:00<?, ?it/s]

Generating projects:   0%|          | 0/10 [00:00<?, ?it/s]

Generating tags:   0%|          | 0/8 [00:00<?, ?it/s]

sqlseed 0.1.16.dev1+g0824e8553.d20260505 | Database: /Users/sunbo/Documents/webblock/sqlseed/examples/sqlseed_demo.db


### 📍 架构定位

| 模块 | 文件 | 核心类/函数 |
|------|------|------------|
| 数据流生成 | `src/sqlseed/generators/stream.py` | `DataStream` |
| 依赖排序 | `src/sqlseed/core/column_dag.py` | `ColumnDAG` |
| 约束回溯 | `src/sqlseed/core/constraints.py`| `ConstraintSolver` |

> 对应架构图: [§6 列依赖 DAG 与约束回溯](../docs/architecture.zh-CN.md#6-列依赖-dag-与约束回溯)

## 1. 先看效果 — 派生列的力量

很多时候，列之间存在依赖关系：`short_code` 是 `project_no` 的后 6 位，`description` 由 `project_no` 拼接而成。

sqlseed 的 `derive_from` + `expression` 让你声明这种关系，**自动生成正确的派生值**：

In [2]:

with connect(str(db_path)) as orch:
    preview = orch.preview_table('projects', count=5, columns={
        'project_no': {'generator': 'pattern', 'params': {'pattern': PRJ_PATTERN}},
        'short_code': {'derive_from': 'project_no', 'expression': 'value[-6:]'},
        'description': {'derive_from': 'project_no', 'expression': "'Project-' + value"},
    })
    print(f"{'project_no':<15s}  {'short_code':<10s}  {'description':<30s}")
    print('-' * 60)
    for row in preview:
        print(f"{row['project_no']:<15s}  {row['short_code']:<10s}  {row['description']:<30s}")

project_no       short_code  description                   
------------------------------------------------------------
PRJ-993498       993498      Project-PRJ-993498            
PRJ-651149       651149      Project-PRJ-651149            
PRJ-697372       697372      Project-PRJ-697372            
PRJ-021699       021699      Project-PRJ-021699            
PRJ-822650       822650      Project-PRJ-822650            


只需声明依赖关系，sqlseed 自动：

1. **拓扑排序** — 确定生成顺序（先 `project_no`，再 `short_code` 和 `description`）
2. **表达式求值** — 使用 `simpleeval` 安全引擎执行表达式
3. **超时保护** — 5 秒超时防止无限循环

下面详细拆解每个机制。

## 2. derive_from + expression

当一列的值依赖另一列时，使用 `derive_from` 声明依赖关系，`expression` 指定计算表达式：

```yaml
columns:
  - name: project_no
    generator: pattern
    params:
      regex: "PRJ-\\d{6}"
  - name: short_code
    derive_from: project_no
    expression: "value[-6:]"
```

sqlseed 会先生成 `project_no`，再用 `value[-6:]` 计算 `short_code`。

In [3]:
import sqlite3

result = fill(
    str(db_path),
    table="projects",
    count=5, clear_before=True,
    columns={
        "project_no": {"type": "pattern", "regex": PRJ_PATTERN},
        "short_code": {"derive_from": "project_no", "expression": "value[-6:]"},
    },
)


conn = sqlite3.connect(str(db_path))
rows = conn.execute("SELECT project_no, short_code FROM projects").fetchall()
for row in rows:
    print(f"project_no={row[0]}, short_code={row[1]}")
conn.close()

Generating projects:   0%|          | 0/5 [00:00<?, ?it/s]

project_no=PRJ-291676, short_code=291676
project_no=PRJ-225564, short_code=225564
project_no=PRJ-093282, short_code=093282
project_no=PRJ-893558, short_code=893558
project_no=PRJ-354600, short_code=354600


## 3. ColumnDAG 拓扑排序

当存在多级依赖时，`ColumnDAG` 自动进行拓扑排序，确保生成顺序正确：

```
project_no → short_code (value[-6:])
           → description (concat('Project ', value))
```

如果存在循环依赖，sqlseed 会抛出 `CyclicDependencyError`。

In [4]:
result = fill(
    str(db_path),
    table="projects",
    count=5,
    clear_before=True,
    columns={
        "project_no": {"type": "pattern", "regex": PRJ_PATTERN},
        "short_code": {"derive_from": "project_no", "expression": "value[-6:]"},
        "description": {"derive_from": "project_no", "expression": "concat('Project ', value)"},
    },
)

conn = sqlite3.connect(str(db_path))
rows = conn.execute("SELECT project_no, short_code, description FROM projects").fetchall()
for row in rows:
    print(f"project_no={row[0]}, short_code={row[1]}, description={row[2]}")
conn.close()

Generating projects:   0%|          | 0/5 [00:00<?, ?it/s]

project_no=PRJ-349829, short_code=349829, description=Project PRJ-349829
project_no=PRJ-133393, short_code=133393, description=Project PRJ-133393
project_no=PRJ-308807, short_code=308807, description=Project PRJ-308807
project_no=PRJ-107130, short_code=107130, description=Project PRJ-107130
project_no=PRJ-848690, short_code=848690, description=Project PRJ-848690


## 4. ExpressionEngine 21 个安全函数

表达式引擎基于 `simpleeval`，提供 21 个安全函数：

| 类别 | 函数 | 说明 |
|------|------|------|
| 字符串 | `upper`, `lower`, `strip`, `lstrip`, `rstrip` | 大小写/去空格 |
| 字符串 | `replace`, `lpad`, `rpad` | 替换/填充 |
| 字符串 | `substring`, `concat` | 截取/拼接 |
| 数学 | `abs`, `round`, `ceil`, `floor` | 取整 |
| 数学 | `min`, `max`, `clamp` | 极值/限制 |
| 转换 | `int`, `float`, `str`, `len` | 类型转换 |
| 哈希 | `md5`, `sha256` | 哈希摘要 |

In [5]:
result = fill(
    str(db_path),
    table="projects",
    count=3,
    clear_before=True,
    columns={
        "project_no": {"type": "pattern", "regex": PRJ_PATTERN},
        "short_code": {"derive_from": "project_no", "expression": "upper(value[-6:])"},
        "name": {"derive_from": "project_no", "expression": "concat('Project-', value)"},
    },
)

conn = sqlite3.connect(str(db_path))
rows = conn.execute("SELECT project_no, short_code, name FROM projects").fetchall()
for row in rows:
    print(f"project_no={row[0]}, short_code={row[1]}, name={row[2]}")
conn.close()

Generating projects:   0%|          | 0/3 [00:00<?, ?it/s]

project_no=PRJ-599719, short_code=599719, name=Project-PRJ-599719
project_no=PRJ-026434, short_code=026434, name=Project-PRJ-026434
project_no=PRJ-417146, short_code=417146, name=Project-PRJ-417146


## 5. ExpressionTimeoutError 超时保护

表达式执行有 5 秒超时保护。如果表达式执行超过 5 秒，会抛出 `ExpressionTimeoutError`。

这防止了恶意或错误的表达式导致无限循环。

## 6. UNIQUE 约束回溯求解

当列有 UNIQUE 约束时，`ConstraintSolver` 使用回溯算法确保不重复：

1. 生成一个值
2. 检查是否与已有值冲突
3. 如果冲突，重新生成（最多重试 N 次）
4. 如果所有重试都失败，回溯到上一行重新生成

demo 数据库中的 UNIQUE 列：
- `members.member_no` (UNIQUE)
- `members.email` (UNIQUE)
- `projects.project_no` (UNIQUE)
- `projects.short_code` (UNIQUE INDEX)
- `tags.name` (UNIQUE)

In [6]:
result = fill(str(db_path), table="members", count=50, clear_before=True)

conn = sqlite3.connect(str(db_path))
member_nos = [r[0] for r in conn.execute("SELECT member_no FROM members").fetchall()]
emails = [r[0] for r in conn.execute("SELECT email FROM members").fetchall()]
print(f"Total rows: {len(member_nos)}")
print(f"Unique member_nos: {len(set(member_nos))}")
print(f"Unique emails: {len(set(emails))}")
print(f"All member_nos unique: {len(member_nos) == len(set(member_nos))}")
print(f"All emails unique: {len(emails) == len(set(emails))}")
conn.close()

Generating members:   0%|          | 0/50 [00:00<?, ?it/s]

Total rows: 50
Unique member_nos: 50
Unique emails: 50
All member_nos unique: True
All emails unique: True


## 7. 概率模式（>100K 行）

当生成超过 100K 行时，回溯求解的性能会下降。`ConstraintSolver` 自动切换到概率模式：

- 使用 SHA256 哈希确保唯一性
- 不需要回溯，性能稳定
- 极小概率碰撞（可忽略）

详见 architecture.md §6 ConstraintSolver

## 🔗 多级 derive_from 链式依赖

`derive_from` 支持链式依赖：A → B → C，ColumnDAG 会自动进行拓扑排序。

In [7]:
with sqlseed.connect(str(db_path)) as orch:
    preview = orch.preview_table('projects', count=3, columns={
        'project_no': {'generator': 'pattern', 'params': {'pattern': PRJ_PATTERN}},
        'short_code': {'derive_from': 'project_no', 'expression': 'value[-6:]'},
        'description': {'derive_from': 'short_code', 'expression': "'Project-' + value"},
    })
    print('多级 derive_from 链 (project_no -> short_code -> description):')
    for row in preview:
        print(f"  {row['project_no']} -> {row['short_code']} -> {row['description']}")

多级 derive_from 链 (project_no -> short_code -> description):
  PRJ-173425 -> 173425 -> Project-173425
  PRJ-923596 -> 923596 -> Project-923596
  PRJ-149663 -> 149663 -> Project-149663


## ⚠️ 实战：ExpressionTimeoutError

表达式执行超过 5 秒会触发超时保护。

In [8]:
from sqlseed.core.expression import ExpressionEngine

engine = ExpressionEngine(timeout_seconds=1)  # 1 second for demo

# Safe expressions execute instantly
result = engine.evaluate('abs(-42)', {})
print(f'abs(-42) = {result}')

# Demonstrate timeout API
print(f'\nExpressionEngine timeout={engine._timeout}s')
print('  - Expressions exceeding the timeout raise ExpressionTimeoutError')
print('  - Default timeout: 5 seconds')
print('  - Used internally by derive_from expressions')
print('  - Thread-based: cannot be killed, only detected')


abs(-42) = 42

ExpressionEngine timeout=1s
  - Expressions exceeding the timeout raise ExpressionTimeoutError
  - Default timeout: 5 seconds
  - Used internally by derive_from expressions
  - Thread-based: cannot be killed, only detected


In [9]:
from sqlseed.core.mapper import ColumnMapper
from sqlseed.core.unique_adjuster import UniqueAdjuster

adjuster = UniqueAdjuster(ColumnMapper())

# Simulate: min_length=2, max_length=2 can only generate ~3844 unique values
# For 50000 rows, UniqueAdjuster increases max_length automatically
print('UniqueAdjuster 自动调整示例:')
print('  输入: min_length=2, max_length=2, count=50000')
print('  输出: max_length 自动扩大以满足 UNIQUE 约束')
print()

# Actual fill with UNIQUE constraint
result = fill(str(db_path), table='members', count=50, clear_before=True, seed=42)
print(f'Filled {result.count} members with UNIQUE member_no')

conn = sqlite3.connect(str(db_path))
sample = conn.execute('SELECT member_no FROM members LIMIT 5').fetchall()
print(f'Sample member_no values: {[r[0] for r in sample]}')
conn.close()

UniqueAdjuster 自动调整示例:
  输入: min_length=2, max_length=2, count=50000
  输出: max_length 自动扩大以满足 UNIQUE 约束



Generating members:   0%|          | 0/50 [00:00<?, ?it/s]

Filled 50 members with UNIQUE member_no
Sample member_no values: [' CI3 dXRZv7', '07gmhZRnFyy5r2xJ7', '1XcW9aTMX1', '4jKEIQOkrtDX', '7gFp6r7O4-25u85HFJ E']


## 🔐 复合唯一约束

`ConstraintSolver` 支持多列联合唯一约束，确保组合值不重复。

In [10]:
from sqlseed.core.constraints import ConstraintSolver

solver = ConstraintSolver()

# Register composite unique (org_code, member_no)
pairs = set()
for i in range(20):
    org = f'ORG-{i % 3}'
    member = f'M{i:04d}'
    ok = solver.check_and_register_composite('org_member', (org, member))
    pairs.add((org, member))

print(f'Registered {len(pairs)} unique (org_code, member_no) pairs')
print(f'Sample: {list(pairs)[:3]}')

Registered 20 unique (org_code, member_no) pairs
Sample: [('ORG-0', 'M0012'), ('ORG-1', 'M0013'), ('ORG-2', 'M0017')]


## 总结

| 功能 | 说明 |
|------|------|
| derive_from | 声明列依赖关系 |
| expression | 计算派生值（基于 simpleeval） |
| ColumnDAG | 拓扑排序确定生成顺序 |
| ExpressionEngine | 21 个安全函数，5 秒超时保护 |
| ConstraintSolver | UNIQUE 回溯求解 + 概率模式 |
| UniqueAdjuster | 自动扩大长度以满足唯一约束 |
| 复合唯一约束 | 多列联合唯一 |

**下一步**: [06-config-deep-dive.ipynb](06-config-deep-dive.ipynb) — 配置模型深度解析

In [11]:
# ✅ 验证: 确保数据已成功生成并写入
import sqlite3
conn = sqlite3.connect(str(db_path))
try:
    # 基本行数验证
    member_count = conn.execute("SELECT COUNT(*) FROM members").fetchone()[0]
    assert member_count > 0, f"Expected members > 0, got {member_count}"
    print("✅ All assertions passed")
finally:
    conn.close()

✅ All assertions passed
